# 聚类算法

在人类眼中，我们很容易识别一些物质世界存在的某些事物的特征，例如：看到在水里的生物，其中有一类是鱼，主要特征是：
1. 通常生存在水中；
2. 是脊椎动物等。

通过对这两个共性特征的概括和归纳，人们提出了鱼这一概念，这使得"鱼类"可以从在水里生存的其他生物中区分开来。

对于机器学习而言，我们希望计算机通过传入一些真实世界客观存在的信息（数据），帮助我们对有相同特征的对象进行归纳和划分，使得我们能去洞察那些可能存在的不同类型，以便于我们能对客观现象做出更深一步的认识和判断。

聚类（cluster）算法在机器学习中有若干种，本节课我们将讲述K-means 算法、层次聚类和 DBSCAN 这三种算法。

## 1. K-means 算法原理与步骤
K-means 是一种**无监督学习**算法，它不需要事先知道数据应该分成几类，也不需要任何标签，仅根据数据本身的特征就能自动分组。

K-means 的目标是将数据集划分为K个簇（clusters），使得每个数据点属于距离最近的簇中心。通过反复调整簇中心的位置，K-means 不断优化簇内的紧密度，从而获得尽量紧凑、彼此分离的簇。

**核心思想**

- 簇（Cluster）：K-means 通过最小化簇内距离的平方和，使得数据点在簇内聚集。一个簇是数据点的集合，这些点在某种意义上“彼此相似”。比如，可以将商场顾客分为“学生群体”“上班族”“退休老人”这三个簇。
- 簇中心（Centroid）：簇中心是簇中所有点的平均值，表示簇的中心位置。
- 簇分配和更新：K-means 通过反复迭代，调整簇的分配，使得簇内数据点与质心的距离尽可能小，逐步收敛。






如下图：      
以簇中心为中心，划分范围
<div class='insertContainerBox column'>
    <div class='insertItem' align=left><img src="https://imgbed.momodel.cn/hv/20250804103617755.png" width="800px"/></div>
</div>

### 1.1 算法工作原理的通俗解释

我们可以把 K-means 算法比作班主任给学生分组：

1. 班主任随机指定3个学生作为组长（初始质心）
2. 其他学生计算与每个组长的距离，选择最近的组长加入
3. 每组重新计算中心位置（新组长）
4. 重复这个过程，直到组长位置不再变化

在实际应用中，这个"距离"通常是欧几里得距离（直线距离），就像地图上计算两个地点的距离一样。

### 1.2 K-means 算法的详细步骤
让我们用一个电商用户分群的案例，一步步拆解 K-means 的工作流程。
```mermaid
flowchart TD
    A[开始] --> B[数据准备: 标准化特征]
    B --> C[初始化: 随机选择K个质心]
    C --> D[分配步骤: 将每个点分配到最近质心的簇]
    D --> E[更新步骤: 重新计算每个簇的质心]
    E --> F{收敛判断}
    F -- 否 --> D
    F -- 是 --> G[输出最终聚类结果]
    
```

**1. 数据准备**

假设我们有某电商平台的用户消费数据，包含两个特征：
- 月均消费金额（单位：元）
- 月均消费频次（单位：次）

原始数据可能长这样：

| 用户ID | 月均消费金额 | 月均消费频次 |
|--------|--------------|--------------|
| 1      | 1500         | 8            |
| 2      | 300          | 2            |
| ...    | ...          | ...          |

**重要提醒**：不同特征的单位和量纲可能差异很大（比如金额是几千，频次是个位数），直接计算距离会导致量纲大的特征主导结果。所以必须先做标准化处理。

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
data_scaled = scaler.fit_transform(raw_data[['月均消费金额','月均消费频次']])
```

标准化后，所有特征都会变成均值为0、标准差为1的分布，消除了量纲影响。

**2. 初始化质心**

K-means需要预先指定簇的数量 K。在我们的案例中，假设市场部希望将用户分为 3 类（高、中、低价值），所以 K=3。

算法会随机选择 3 个点作为初始质心。在scikit-learn中，可以通过`random_state`参数控制随机种子，确保结果可复现。

```python
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(data_scaled)
```

**3. 分配数据点到最近质心**

对于每个数据点，计算它与所有质心的距离，并将其分配到最近的质心所在的簇。欧式距离的计算公式为：

$$
\text{距离} = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}
$$

这个公式其实就是在二维平面上计算两点间的直线距离，就像用尺子在地图上量距离一样。

**4. 重新计算质心**

每个簇的新质心是该簇所有点的平均值。例如，如果一个簇有5个点，它们的坐标分别是(1,2), (1,4), (2,3), (3,5), (4,4)，那么新质心的坐标就是：


$x = (1+1+2+3+4)/5 = 2.2$

$y = (2+4+3+5+4)/5 = 3.6$

**5. 迭代直到收敛**

重复步骤 3 和 4，直到满足以下任一条件：
- 质心的位置变化小于某个阈值（默认 0.0001）
- 达到最大迭代次数（默认 300 次）
- 所有数据点所属的簇不再变化



In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 模拟一些原始数据
data = {
    '用户ID': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    '月均消费金额': [1500, 300, 2000, 500, 800, 1200, 1000, 2500, 1800, 700],
    '月均消费频次': [8, 2, 10, 1, 3, 6, 5, 12, 9, 4]
}

# 将数据转换为DataFrame
raw_data = pd.DataFrame(data)

# 数据准备：标准化处理
scaler = StandardScaler()
data_scaled = scaler.fit_transform(raw_data[['月均消费金额', '月均消费频次']])

# 初始化K-means算法
kmeans = KMeans(n_clusters=3, random_state=42)

# 训练模型并预测簇
clusters = kmeans.fit_predict(data_scaled)

# 将簇标签添加到原始数据中
raw_data['簇'] = clusters

# 输出标准化后的数据和原始数据与簇标签
print("标准化后的数据：")
print(data_scaled)
print("\n原始数据与簇标签：")
print(raw_data)

# 打印质心坐标
print("\n质心坐标（标准化后的特征空间）：")
print(kmeans.cluster_centers_)

# 可视化结果
plt.figure(figsize=(10, 6))
# 绘制用户数据点
plt.scatter(data_scaled[:, 0], data_scaled[:, 1], c=clusters, cmap='viridis', marker='o', label='用户')
# 绘制质心
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c='red', marker='x', s=100, label='质心')
# 添加文本标签显示质心坐标
for i, center in enumerate(kmeans.cluster_centers_):
    plt.text(center[0], center[1], f'({center[0]:.2f}, {center[1]:.2f})', fontsize=12, color='red')
plt.title('K-means用户分群')
plt.xlabel('标准化后的月均消费金额')
plt.ylabel('标准化后的月均消费频次')
plt.legend()
plt.show()

运行上面这段代码，我们可以看到用户被分为3个簇，并且能看到每个簇的质心。

### 1.3 K 值的选择
确定最佳的簇数 K 是 K-means 聚类中的一个难点。常用的选择方法有：

1. **肘部法则(Elbow Method)**：
   - 原理：随着K值增大，误差平方和(SSE)会减小，但减幅会逐渐变缓
   - 操作：计算不同K值时的SSE，绘制曲线，选择"拐点"（像肘部一样的点）
```python
sse = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(data_scaled)
    sse.append(kmeans.inertia_)  # SSE值

plt.plot(range(1, 11), sse, marker='o')
plt.xlabel('K值')
plt.ylabel('SSE')
plt.show()
```

下图可以发现当 k=4 或者 5 时是最佳的情况 SSE 图像下降幅度最大放缓的情况在 4-5 之间。
<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804104255157.png" width="800px"/></div>
</div>


2. **轮廓系数(Silhouette Coefficient)**：
   - 范围：-1 到 1，值越大表示聚类效果越好
   - 计算：对于每个点，计算它与同簇其他点的平均距离(a)，与最近其他簇的平均距离$(b)$，轮廓系数为$(b-a)/max(a,b)$
  
```python
from sklearn.metrics import silhouette_score

silhouette_scores = []
for k in range(2, 11):
   kmeans = KMeans(n_clusters=k, random_state=42)
   cluster_labels = kmeans.fit_predict(data_scaled)
   silhouette_scores.append(silhouette_score(data_scaled, cluster_labels))

plt.plot(range(2, 11), silhouette_scores, marker='o')
plt.xlabel('K值')
plt.ylabel('轮廓系数')
plt.show()
```

例如下图我们在 K=6 时就能获得最大的轮廓系数，说明这时候聚类效果最好。

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804104633887.png" width="600px"/></div>
</div>



### 1.4 K-means 的优缺点分析

**优点**

1. **简单高效**：原理直观，计算复杂度为O(n)，适合大规模数据
2. **易于实现**：主流工具包都有现成实现
3. **结果可解释**：簇中心可以代表该簇的"典型"特征

**局限**

1. **需要预先指定K值**：实际中最佳 K 值往往难以确定
2. **对初始质心敏感**：可能导致局部最优解
3. **假设簇是凸形的**：难以处理复杂形状的簇
4. **对噪声和离群点敏感**：可能扭曲质心位置

## 2. 层次聚类与 DBSCAN 简介
层次聚类和 DBSCAN 是两种常用的聚类算法，它们都能帮助我们发现数据中的自然分组。想象一下，你有一堆杂乱无章的积木，层次聚类就像是从下往上把这些积木一层层堆叠起来，而 DBSCAN 则像是用磁铁把相互靠近的积木吸在一起，同时把孤立的积木单独放在一边。

这两种算法在实际中有广泛应用。比如在电商领域，可以用它们来分析用户行为模式；在物流行业，可以帮助优化配送站点的布局；在金融领域，还能用来检测异常交易。接下来，我们将深入探讨这两种算法的原理和应用。

### 2.1 层次聚类


#### 2.1.1 层次聚类的基本原理
层次聚类(Hierarchical Clustering)是一种通过构建树状图(dendrogram)来展示数据层次结构的聚类方法。它有两种主要方式：
1. **凝聚式(自底向上)（Agglomerative 层次聚类）**：开始时将每个数据点视为一个簇，然后逐步合并最相似的簇
2. **分裂式(自顶向下)（Divisive 层次聚类）**：开始时将所有数据点视为一个簇，然后逐步分裂


<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804110032292.png" width="600px"/></div>
</div>

在实际应用中，凝聚式层次聚类更为常见。它的工作过程就像是在组织一场社交活动：最初每个人都是独立的，然后根据相似度(比如共同爱好)两两配对，接着这些小组再合并成更大的团体，最终形成一个完整的社交网络。

#### 2.1.2 树状图的解读方法
树状图是层次聚类的可视化工具，它展示了数据点如何被逐步合并。解读树状图时要注意三个关键要素：

1. **y轴高度**：表示两个簇被合并时的距离或相异度。高度越高，说明被合并的簇差异越大。
2. **横轴顺序**：数据点的排列顺序会影响树状图的形状，但不影响聚类结果。
3. **切割高度**：通过在特定高度画一条水平线，可以决定最终的簇数量。



#### 2.1.3 自顶向下的层次聚类算法(Divisive)


Hierarchical K-means算法是“自顶向下”的层次聚类算法，用到了基于划分的聚类算法那 K-means，算法思路如下：

首先，把原始数据集放到一个簇 C，这个簇形成了层次结构的最顶层      
使用 K-means 算法把簇 C 划分成指定的 K 个子簇


$$ C_i, i = 1, 2, \ldots, k $$

形成一个新的层

对于步骤 2 所生成的 K 个簇，递归使用 K-means 算法划分成更小的子簇，直到每个簇不能再划分（只包含一个数据对象）或者满足设定的终止条件。
如下图，展示了一组数据进行了二次 K-means 算法的过程：

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804124927479.png" width="800px"/></div>
</div>

Hierarchical K-means 算法一个很大的问题是，一旦两个点在最开始被划分到了不同的簇，即使这两个点距离很近，在后面的过程中也不会被聚类到一起。


<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804124927478.png" width="800px"/></div>
</div>

对于以上的例子，红色椭圆框中的对象聚类成一个簇可能是更优的聚类结果，但是由于橙色对象和绿色对象在第一次K-means就被划分到不同的簇，之后也不再可能被聚类到同一个簇。

Bisecting k-means 聚类算法，即二分 k 均值算法，是分层聚类（Hierarchical clustering）的一种。


#### 2.1.4 自底向上的层次聚类算法(Agglomerative)

层次聚类的合并算法通过计算两类数据点间的相似性，对所有数据点中最为相似的两个数据点进行组合，并反复迭代这一过程。简单的说层次聚类的合并算法是通过计算每一个类别的数据点与所有数据点之间的距离来确定它们之间的相似性，距离越小，相似度越高。并将距离最近的两个数据点或类别进行组合，生成聚类树。

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804133824196.png" width="800px"/></div>
</div>

相比于 Hierarchical K-means 算法存在的问题，Agglomerative Clustering 算法能够保证距离近的对象能够被聚类到一个簇中，该算法采用的“自底向上”聚类的思路

**Agglomerative 算法示例**

对于如下数据：

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804133956340.png" width="400px"/></div>
</div>

1. 将A到F六个点，分别生成6个簇
2. 找到当前簇中距离最短的两个点，这里我们使用单连锁的方式来计算距离，发现A点和B点距离最短，将A和B组成一个新的簇，此时簇列表中包含五个簇，分别是{A，B}，{C}，{D}，{E}，{F}，如下图所示：

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804133956339.png" width="400px"/></div>
</div>

3. 重复步骤2、发现{C}和{D}的距离最短，连接之，然后是簇{C,D}和簇{E}距离最短，依次类推，直到最后只剩下一个簇，得到如下所示的示意图：

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804133956338.png" width="400px"/></div>
</div>

4. 此时原始数据的聚类关系是按照层次来组织的，选取一个簇间距离的阈值，可以得到一个聚类结果，比如在如下红色虚线的阈值下，数据被划分为两个簇：簇{A，B，C，D，E}和簇{F}

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804133956337.png" width="400px"/></div>
</div>

Agglomerative 聚类算法的优点是能够根据需要在不同的尺度上展示对应的聚类结果，缺点同 Hierarchical K-means 算法一样，一旦两个距离相近的点被划分到不同的簇，之后也不再可能被聚类到同一个簇，即无法撤销先前步骤的工作。另外，Agglomerative 性能较低，并且因为聚类层次信息需要存储在内存中，内存消耗大，不适用于大量级的数据聚类。

#### 2.1.5 Python 示例
让我们通过一个电商用户分群的案例来学习如何用Python实现层次聚类。我们将使用`RFM`模型数据(最近购买时间`Recency`、购买频率`Frequency`、消费金额`Monetary`)。

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import matplotlib.pyplot as plt
import pandas as pd

# 示例数据
rfm_data = np.array([
    [10, 5, 200],  # 用户1
    [20, 3, 150],  # 用户2
    [5, 10, 300],  # 用户3
    [15, 8, 250],  # 用户4
    [8, 4, 180]    # 用户5
])

user_ids = ['用户1', '用户2', '用户3', '用户4', '用户5']

# 计算距离矩阵
Z = linkage(rfm_data, method='ward')  # 使用Ward方法计算簇间距离

# 绘制树状图
plt.figure(figsize=(12, 6))
dendrogram(Z, labels=user_ids, leaf_rotation=90)
plt.title('电商用户层次聚类树状图')
plt.xlabel('用户ID')
plt.ylabel('距离')
plt.show()

# 切割树状图获取聚类结果
clusters = fcluster(Z, t=3, criterion='maxclust')  # 分为3个簇
print("聚类结果：", clusters)

# 将数据转换为DataFrame
rfm_df = pd.DataFrame(rfm_data, columns=['最近购买时间', '购买频率', '消费金额'])
rfm_df['Cluster'] = clusters

# 计算每个簇的平均RFM值
cluster_summary = rfm_df.groupby('Cluster').mean()
print("每个簇的平均RFM值：")
print(cluster_summary)

通过层次聚类，我们将电商用户分为 3 个簇，并计算了每个簇的平均 RFM 值。每个簇的特征如下：
- 簇 1：活跃用户，有一定的消费能力。（用户 1、2、5）
- 簇 2：流失风险较高的用户，需要关注并采取措施挽回。（用户 3）
- 簇 3：高价值用户，需要重点关注并维护。（用户 4）


这段代码中，`linkage()`函数是关键，它计算并返回聚类过程。`method='ward'`表示使用 Ward 方差最小化算法，这是最常用的方法之一。其他可选方法包括`single`(最小距离)、`complete`(最大距离)和`average`(平均距离)。

#### 2.1.6 层次聚类的优缺点

**优点**：
1. 可视化直观：树状图能清晰展示数据层次结构
2. 不需要预先指定簇数量
3. 可以发现任意形状的簇

**缺点**：
1. 计算复杂度高：不适合大数据集(O(n³)时间复杂度)
2. 对噪声和异常值敏感
3. 一旦合并或分裂就不能撤销

在实际应用中，当数据量超过几千条时，层次聚类的计算就会变得非常缓慢。这时可以考虑先使用K-means进行预处理，再对K-means的中心点进行层次聚类。

### 2.2 DBSCAN


#### 2.2.1 DBSCAN 算法原理
DBSCAN(Density-Based Spatial Clustering of Applications with Noise)是一种基于密度的聚类算法。与层次聚类不同，它特别擅长发现任意形状的簇并识别噪声点。

DBSCAN 的核心思想很简单：物以类聚。它定义了两个重要参数：
1. **ε(eps)**：邻域半径，决定了一个点的"个人空间"大小
2. **MinPts**：最小点数，形成一个簇所需的最少"朋友"数量

算法将点分为三类：
1. **核心点**：在 ε 半径内有至少 MinPts 个邻居的点
2. **边界点**：在 ε 半径内邻居少于 MinPts，但属于某个核心点的邻域
3. **噪声点**：既不是核心点也不是边界点的点

#### 2.2.2 DBSCAN参数选择技巧
选择合适的 ε 和 MinPts 是 DBSCAN 成功的关键。以下是几种实用的参数确定方法：

1. **K 距离图法**：
   - 计算每个点到其第 k 近邻的距离
   - 将这些距离排序后绘图
   - 选择图中拐点对应的距离作为ε
   - 通常 k 取 MinPts-1

In [ ]:
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import numpy as np

# 随机生成一个包含 100 个点，每个点有 2 个特征的数据集
np.random.seed(0)  # 设置随机种子以保证结果可复现
X = np.random.rand(100, 2)  # 100x2 的数组

# 创建 NearestNeighbors 实例并拟合数据
neigh = NearestNeighbors(n_neighbors=5)
nbrs = neigh.fit(X)

# 查找每个点的 5 个最近邻居的距离和索引
distances, indices = nbrs.kneighbors(X)

# 由于每个点的最近邻居中包括它自己，我们取除了第一个（自己）之外的第五个最近邻居的距离
# 如果距离数组中的距离数量小于5，则取最后一个
distances = np.sort(distances[:, 1:], axis=0)[:, -1]

# 绘制第五最近邻居的距离图
plt.plot(distances)
plt.title('K距离图(k=5)')
plt.xlabel('点数')
plt.ylabel('距离')
plt.show()

- 图表显示：图表显示了每个点到其第 5 个最近邻居的距离随点的索引的变化情况。
    - 如果距离随着点的索引增加而 平稳增加，这表明数据点在空间中分布较为均匀。
    - 如果距离波动较大，这表明数据点的分布不均匀，有些区域点更密集，有些区域点更稀疏。
- 选择 ε：
    - 在 K 距离图中，找到一个明显的 拐点。拐点之前的距离较小，表示这些点周围的邻居较多，属于高密度区域；拐点之后的距离较大，表示这些点周围的邻居较少，属于低密度区域。
    - 拐点对应的距离可以作为 ε 的值，因为这个距离能够较好地区分高密度区域和低密度区域。

2. **经验法则**：
   - 对于二维数据，MinPts 可以从 4 开始尝试
   - 维度每增加 1，MinPts 大致增加 1 倍
   - ε 通常选择使每个簇包含 1-2% 的数据点

3. **网格搜索**：
   - 在一定范围内测试不同参数组合
   - 选择产生最稳定结果的参数


#### 2.2.3 DBSCAN 示例：信用卡异常检测
让我们用 DBSCAN 来解决一个实际问题：检测信用卡异常交易。假设我们有以下特征：
- 交易金额
- 交易时间(转换为一天中的分钟数)
- 交易地点的经纬度

In [ ]:
# 导入必要的库
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

# 1. 生成示例数据
# 使用make_moons生成两个月牙形的数据集，添加一些噪声
X, _ = make_moons(n_samples=300, noise=0.05, random_state=42)

# 2. 可视化原始数据
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], s=10)
plt.title("原始数据")

# 3. 应用DBSCAN算法
# 初始化DBSCAN对象
# eps: 邻域半径
# min_samples: 核心点所需的最小邻域样本数
dbscan = DBSCAN(eps=0.2, min_samples=5)

# 拟合数据并进行聚类
clusters = dbscan.fit_predict(X)

# 4. 可视化聚类结果
plt.subplot(1, 2, 2)

# 获取所有唯一的聚类标签
unique_labels = set(clusters)

# 为每个聚类分配不同的颜色
colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]

for k, col in zip(unique_labels, colors):
    if k == -1:
        # 噪声点（未聚类点）显示为黑色
        col = [0, 0, 0, 1]
    
    # 获取当前标签的所有点
    class_member_mask = (clusters == k)
    
    # 绘制当前聚类的点
    xy = X[class_member_mask]
    plt.scatter(xy[:, 0], xy[:, 1], s=10, c=[col], label=f'Cluster {k}')

plt.title(f"DBSCAN聚类结果\n发现{len(unique_labels)-1}个聚类")
plt.legend()
plt.tight_layout()
plt.show()

# 5. 打印聚类信息
n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
n_noise = list(clusters).count(-1)

print(f"估计的聚类数量: {n_clusters}")
print(f"噪声点数量: {n_noise}")


1. **数据生成**：使用`make_moons`生成两个月牙形状的数据集，这是测试聚类算法的经典数据集。

2. **DBSCAN参数**：
   - `eps`：邻域半径，控制两个样本被视为邻居的最大距离
   - `min_samples`：核心点所需的最小邻域样本数

3. **聚类结果**：
   - 不同颜色代表不同的聚类
   - 黑色点代表噪声点（未分配到任何聚类的点）

4. **输出信息**：显示发现的聚类数量和噪声点数量


#### 2.2.4 DBSCAN的优缺点

**优点**：
1. 不需要预先指定簇数量
2. 能发现任意形状的簇
3. 对噪声和异常值鲁棒
4. 只需两个参数(ε 和 MinPts)

**缺点**：
1. 对参数敏感，特别是当簇密度差异大时
2. 高维数据中距离度量可能失效(维度灾难)
3. 不适用于密度差异大的数据集

## 3. 聚类算法的选择与评估
在实际应用中，聚类分析可以帮助我们：

1. **发现数据中的自然分组**：比如根据顾客的购买行为划分不同的消费群体
2. **识别异常值**：比如在信用卡交易中发现异常消费模式
3. **数据预处理**：为其他机器学习算法准备数据
4. **简化复杂数据**：通过分组使大数据集更易于理解和处理

### 3.1 为什么需要选择合适的聚类算法？
不同的聚类算法就像不同的工具 - 螺丝刀、锤子、钳子各有各的用途。选择错误的算法可能导致：

- 分组结果不符合实际情况
- 计算资源浪费
- 难以解释的结果
- 错过重要的数据模式

举个实际例子：假设我们要对城市进行气候区域划分。如果使用 K-means 算法（假设 k=3），可能会得到三个明显的区域。但如果使用 DBSCAN 算法，可能会发现一些特殊的"微气候"区域，这些区域虽然小但气候特征明显不同于周边地区。两种算法各有优劣，取决于我们的具体需求。

### 3.2 聚类评估方法

#### 3.2.1 内部评估指标

1. 轮廓系数(Silhouette Coefficient)

轮廓系数衡量一个对象与自身簇的相似度相比与其他簇的相似度。取值范围[-1,1]，值越大表示聚类越好。

2. 肘部法则(Elbow Method)

肘部法则用于确定K-means中的最佳k值。原理是观察不同k值下总平方误差(SSE)的变化，选择SSE下降开始变缓的点。


#### 3.2.2 外部评估指标
**调整兰德指数(Adjusted Rand Index)**


衡量两个聚类结果的相似度，取值范围[-1,1]，1表示完全一致。

```python
from sklearn.metrics import adjusted_rand_score

true_labels = [...]  # 真实标签
ari = adjusted_rand_score(true_labels, labels)
print(f"调整兰德指数: {ari:.3f}")
```

### 3.3 聚类算法选择指南

选择聚类算法时，需要考虑以下因素：

1. **数据规模**：
   - 大数据集：K-means、Mini-Batch K-means
   - 小数据集：层次聚类、谱聚类

2. **数据维度**：
   - 高维数据：可能需要先降维（PCA、t-SNE）
   - 低维数据：大多数算法都适用

3. **簇形状**：
   - 球形簇：K-means
   - 任意形状：DBSCAN、谱聚类

4. **噪声和异常值**：
   - 含噪声数据：DBSCAN、OPTICS
   - 干净数据：大多数算法都适用

5. **是否需要自动确定簇数量**：
   - 需要：DBSCAN、层次聚类
   - 不需要：K-means、GMM

```mermaid
graph TD
    A[开始选择聚类算法] --> B{数据规模大?}
    B -->|是| C[考虑K-means或Mini-Batch K-means]
    B -->|否| D{需要自动确定簇数量?}
    D -->|是| E[考虑DBSCAN或层次聚类]
    D -->|否| F{簇形状是球形?}
    F -->|是| G[使用K-means]
    F -->|否| H[考虑DBSCAN或谱聚类]
```

### 3.4 三种算法的对比展示


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs, make_circles
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# 设置随机种子保证结果可复现
np.random.seed(42)

# 创建三种不同形状的数据集
n_samples = 300
noisy_moons = make_moons(n_samples=n_samples, noise=0.05)
blobs = make_blobs(n_samples=n_samples, random_state=8)
noisy_circles = make_circles(n_samples=n_samples, factor=0.5, noise=0.05)

datasets = [
    (noisy_moons, "半月形数据"),
    (blobs, "团状数据"),
    (noisy_circles, "环形数据")
]

# 初始化三种聚类算法
kmeans = KMeans(n_clusters=2)
agglo = AgglomerativeClustering(n_clusters=2, linkage='ward')
dbscan = DBSCAN(eps=0.2, min_samples=5)

# 创建绘图
plt.figure(figsize=(18, 14))
plot_num = 1

for dataset_idx, (dataset, dataset_name) in enumerate(datasets):
    X, y = dataset
    
    # 标准化数据（对DBSCAN很重要）
    X = StandardScaler().fit_transform(X)
    
    # 原始数据分布
    plt.subplot(len(datasets), 4, plot_num)
    plt.scatter(X[:, 0], X[:, 1], s=10, c=y)
    plt.title(f"{dataset_name}\n真实分类")
    plot_num += 1
    
    # 应用三种算法
    algorithms = [
        (kmeans, "K-Means"),
        (agglo, "层次聚类"),
        (dbscan, "DBSCAN")
    ]
    
    for algorithm, algo_name in algorithms:
        # 训练模型
        if algo_name == "DBSCAN" and dataset_idx == 2:  # 为环形数据调整DBSCAN参数
            algorithm.set_params(eps=0.3)
        
        y_pred = algorithm.fit_predict(X)
        
        # 计算轮廓系数（如果可能）
        if algo_name != "DBSCAN" or len(np.unique(y_pred)) > 1:
            silhouette = silhouette_score(X, y_pred)
            title = f"{algo_name}\n轮廓系数: {silhouette:.2f}"
        else:
            title = f"{algo_name}\n轮廓系数: 无法计算"
        
        # 绘制结果
        plt.subplot(len(datasets), 4, plot_num)
        plt.scatter(X[:, 0], X[:, 1], s=10, c=y_pred)
        plt.title(title)
        plot_num += 1

plt.tight_layout()
plt.show()

运行上述代码后，你将看到以下输出：
- 每个数据集的原始分布和三种聚类算法的聚类结果。
- 每个聚类结果旁边会显示轮廓系数的值。
- 如果 DBSCAN 的聚类结果中只有一个聚类或所有点都是噪声，轮廓系数将显示为“无法计算”。

- K-Means 和层次聚类在团状数据集上表现较好，但在半月形和环形数据集上效果一般。
- DBSCAN 在所有数据集上的表现均不如 K-Means 和层次聚类，尤其是在半月形和环形数据集上。

轮廓系数表明，对于不同形状的数据集，选择合适的聚类算法至关重要。K-Means 和层次聚类在处理团状数据时效果较好，而对复杂形状的数据集则需要更复杂的算法或参数调整。

### 3.5 聚类稳定性评估

In [ ]:
from sklearn.metrics import pairwise_distances
from sklearn.metrics.cluster import adjusted_rand_score

# 多次运行K-means
all_labels = []
for _ in range(10):
    kmeans = KMeans(n_clusters=3, random_state=None)  # 不固定随机种子
    labels = kmeans.fit_predict(X_scaled)
    all_labels.append(labels)

# 计算两两之间的相似度
stability_scores = []
for i in range(10):
    for j in range(i+1, 10):
        score = adjusted_rand_score(all_labels[i], all_labels[j])
        stability_scores.append(score)

print(f"平均稳定性得分: {np.mean(stability_scores):.3f}")

- 稳定性得分：稳定性得分是通过比较不同运行得到的聚类结果之间的相似度来评估的。得分范围从 -1 到 1，其中 1 表示完全一致，0 表示随机一致性，-1 表示完全不一致。
- 平均稳定性得分：这个得分表示在多次运行 K-means 算法时，聚类结果的平均一致性。得分越高，表示聚类结果越稳定，即不同运行之间的聚类结果越相似。